# Liu2024 CSP/FBCSP Paper-Match Tuning Notebook

This notebook is meant to narrow the gap between our Python baseline and the Liu2024 paper baseline.

It uses the original Figshare `sourcedata` `.mat` trials, not MOABB windows. The goal is not to blindly maximize accuracy, but to test the implementation choices that are underspecified in the paper:

- time window interpretation: `0–4s`, `2–6s`, shifted 4s windows, and optionally the full 8s trial;
- channel set: 29 EEG channels after removing the reference/EOG/marker vs 30 EEG including the reference channel;
- referencing: none vs average reference;
- baseline removal: no baseline, whole-trial mean removal, pre-window mean removal, or per-window mean removal;
- CSP regularization: no regularization vs Ledoit-Wolf;
- FBCSP: coarse bands vs dense overlapping bands, with optional feature selection.

The output ranks settings by closeness to the paper's reported values:

- CSP + LDA target: **55.57%**
- FBCSP + SVM target: **57.57%**

Run this after you have `sourcedata.zip` extracted or available in `liu2024_figshare/sourcedata`.

## 1. Imports

In [1]:

import os
import re
import json
import zipfile
import warnings
from pathlib import Path
from datetime import datetime
from itertools import product

import numpy as np
import pandas as pd

from scipy import io as scipy_io
from scipy.signal import butter, sosfiltfilt

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    precision_score,
    recall_score,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from mne.decoding import CSP

warnings.filterwarnings('ignore')

## 2. Paths and run controls

In [2]:

# ---- Dataset paths ----
WORKING_DIR = Path.cwd().parent
DATA_ROOT = WORKING_DIR / 'liu2024_figshare'
SOURCE_ZIP_PATH = DATA_ROOT / 'sourcedata.zip'
SOURCE_EXTRACT_DIR = DATA_ROOT / 'sourcedata'

# Optional override if your extracted folder is somewhere else.
# Example: Path('/Users/vadim/.../liu2024_figshare/sourcedata')
LOCAL_SOURCE_DIR_OVERRIDE = None

# ---- Run controls ----
RUN_PROFILE = 'focused'  # 'focused' or 'extended'
RANDOM_STATE = 2026
N_SPLITS = 10
TEST_SIZE = 0.40
FS = 500

# Paper targets from Table 4.
TARGET_CSP_LDA = 0.5557
TARGET_FBCSP_SVM = 0.5757

# Runtime note:
# focused = compact candidate set, usually reasonable to run first.
# extended = larger grid. Use only after focused tells you which direction is promising.

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
ARTIFACT_DIR = WORKING_DIR / 'artifacts' / 'liu2024-paper-match-tuning' / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / 'run.log'

print(f'Artifacts: {ARTIFACT_DIR}')

Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-paper-match-tuning/20260518_190650


## 3. Logger

In [3]:
_log_handle = open(LOG_PATH, 'w', buffering=1)

def log(msg=''):
    text = str(msg)
    print(text)
    _log_handle.write(text + '\n')

log('Liu2024 paper-match tuning run')
log(f'Artifacts: {ARTIFACT_DIR}')
log(f'RUN_PROFILE: {RUN_PROFILE}')

Liu2024 paper-match tuning run
Artifacts: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/artifacts/liu2024-paper-match-tuning/20260518_190650
RUN_PROFILE: focused


## 4. Find or download source `.mat` files

In [4]:

FIGSHARE_ARTICLE_ID = 21679035
FIGSHARE_VERSION = 5


def find_source_mat_files(root: Path):
    root = Path(root)
    return sorted(root.rglob('*.mat'))


def download_figshare_file_by_name(target_name: str, output_path: Path):
    import requests
    api_url = f'https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/versions/{FIGSHARE_VERSION}'
    log(f'Querying Figshare API: {api_url}')
    meta = requests.get(api_url, timeout=30).json()
    files = meta.get('files', [])
    available = [(f.get('name'), f.get('download_url')) for f in files]
    log(f'Figshare files: {[name for name, _ in available]}')
    for name, url in available:
        if name == target_name:
            log(f'Downloading {target_name} -> {output_path}')
            output_path.parent.mkdir(parents=True, exist_ok=True)
            with requests.get(url, stream=True, timeout=60) as r:
                r.raise_for_status()
                with open(output_path, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f.write(chunk)
            return output_path
    raise FileNotFoundError(f'Could not find {target_name} in Figshare files: {available}')


def get_source_dir():
    if LOCAL_SOURCE_DIR_OVERRIDE is not None:
        p = Path(LOCAL_SOURCE_DIR_OVERRIDE)
        if not p.exists():
            raise FileNotFoundError(f'LOCAL_SOURCE_DIR_OVERRIDE does not exist: {p}')
        return p

    if SOURCE_EXTRACT_DIR.exists() and find_source_mat_files(SOURCE_EXTRACT_DIR):
        log(f'Using existing extracted source dir: {SOURCE_EXTRACT_DIR}')
        return SOURCE_EXTRACT_DIR

    if not SOURCE_ZIP_PATH.exists():
        log(f'Local sourcedata.zip not found at {SOURCE_ZIP_PATH}')
        download_figshare_file_by_name('sourcedata.zip', SOURCE_ZIP_PATH)

    log(f'Extracting {SOURCE_ZIP_PATH} -> {SOURCE_EXTRACT_DIR}')
    SOURCE_EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SOURCE_ZIP_PATH, 'r') as zf:
        zf.extractall(SOURCE_EXTRACT_DIR)
    return SOURCE_EXTRACT_DIR


SOURCE_DIR = get_source_dir()
MAT_FILES = find_source_mat_files(SOURCE_DIR)
log(f'Total .mat files found: {len(MAT_FILES)}')
for p in MAT_FILES[:5]:
    log(f'  preview: {p}')

if len(MAT_FILES) == 0:
    raise FileNotFoundError(f'No .mat files found under {SOURCE_DIR}')

Using existing extracted source dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata
Total .mat files found: 50
  preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-01/sub-01_task-motor-imagery_eeg.mat
  preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-02/sub-02_task-motor-imagery_eeg.mat
  preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-03/sub-03_task-motor-imagery_eeg.mat
  preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-04/sub-04_task-motor-imagery_eeg.mat
  preview: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/liu2024_figshare/sourcedata/sourcedata/sub-05/s

## 5. Robust MATLAB `.mat` loader

The Figshare files usually have a top-level `eeg` struct with `rawdata` and `label`, not direct top-level arrays.

In [5]:

def subject_id_from_path(path: Path):
    s = str(path)
    m = re.search(r'sub[-_ ]?(\d{1,2})', s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r'\d+', path.stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f'Cannot infer subject id from {path}')


def _is_mat_struct(obj):
    return hasattr(obj, '_fieldnames')


def _walk_mat(obj, prefix=''):
    items = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k.startswith('__'):
                continue
            items.extend(_walk_mat(v, f'{prefix}.{k}' if prefix else k))
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            items.extend(_walk_mat(v, f'{prefix}.{k}' if prefix else k))
    else:
        items.append((prefix, obj))
    return items


def _normalize_rawdata_shape(arr):
    arr = np.asarray(arr)
    arr = np.squeeze(arr)
    if arr.ndim != 3:
        return None
    # Expected dimensions are some permutation of 40 trials, ~33 channels, 4000 samples.
    shape = arr.shape
    axes = list(range(3))
    trial_axis = int(np.argmin([abs(s - 40) for s in shape]))
    remaining = [a for a in axes if a != trial_axis]
    time_axis = remaining[int(np.argmax([shape[a] for a in remaining]))]
    chan_axis = [a for a in axes if a not in (trial_axis, time_axis)][0]
    return np.transpose(arr, (trial_axis, chan_axis, time_axis))


def load_subject_mat(path: Path, preview=False):
    mat = scipy_io.loadmat(path, squeeze_me=True, struct_as_record=False)
    entries = _walk_mat(mat)

    raw_candidates = []
    label_candidates = []

    for name, obj in entries:
        arr = np.asarray(obj)
        squeezed = np.squeeze(arr)

        norm = None
        if squeezed.ndim == 3:
            norm = _normalize_rawdata_shape(squeezed)
            if norm is not None:
                score = 0
                lname = name.lower()
                if 'raw' in lname or 'data' in lname or 'eeg' in lname:
                    score += 10
                if norm.shape[0] == 40:
                    score += 5
                if 25 <= norm.shape[1] <= 40:
                    score += 3
                if norm.shape[2] >= 1000:
                    score += 3
                raw_candidates.append((score, name, norm))

        if squeezed.ndim in (1, 2) and squeezed.size in (40, 50):
            vals = squeezed.astype(int, copy=False).ravel()
            uniq = set(np.unique(vals).tolist())
            if uniq.issubset({0, 1, 2}) and len(uniq) >= 2:
                score = 0
                lname = name.lower()
                if 'label' in lname or 'class' in lname or 'target' in lname:
                    score += 10
                if squeezed.size == 40:
                    score += 5
                label_candidates.append((score, name, vals))

    if not raw_candidates or not label_candidates:
        keys = [name for name, _ in entries]
        raise KeyError(f'Could not infer rawdata/labels from {path}. Entries: {keys[:50]}')

    raw_candidates.sort(key=lambda x: x[0], reverse=True)
    label_candidates.sort(key=lambda x: x[0], reverse=True)

    raw_score, raw_name, rawdata = raw_candidates[0]
    label_score, label_name, labels = label_candidates[0]

    if preview:
        rows = []
        for name, obj in entries:
            arr = np.asarray(obj)
            rows.append({
                'name': name,
                'type': type(obj).__name__,
                'shape': str(getattr(arr, 'shape', None)),
                'dtype': str(getattr(arr, 'dtype', None)),
            })
        pd.DataFrame(rows).to_csv(ARTIFACT_DIR / 'mat_structure_subject_01.csv', index=False)
        log(f'Selected raw data: {raw_name}, shape={rawdata.shape}')
        log(f'Selected labels:   {label_name}, shape={labels.shape}, counts={dict(zip(*np.unique(labels, return_counts=True)))}')

    return rawdata.astype(np.float64), labels.astype(int).ravel(), raw_name, label_name


subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    X_raw, y_raw, raw_name, label_name = load_subject_mat(p, preview=(sid == 1))
    subjects.append({
        'subject_id': sid,
        'path': p,
        'rawdata_shape': X_raw.shape,
        'label_values': dict(zip(*np.unique(y_raw, return_counts=True))),
        'raw_field': raw_name,
        'label_field': label_name,
    })
subjects = sorted(subjects, key=lambda d: d['subject_id'])
summary_df = pd.DataFrame(subjects)
summary_df.to_csv(ARTIFACT_DIR / 'source_mat_summary.csv', index=False)
log(summary_df[['subject_id', 'rawdata_shape', 'label_values']].to_string(index=False))

Selected raw data: eeg.rawdata, shape=(40, 33, 4000)
Selected labels:   eeg.label, shape=(40,), counts={np.int64(1): np.int64(20), np.int64(2): np.int64(20)}
 subject_id  rawdata_shape   label_values
          1 (40, 33, 4000) {1: 20, 2: 20}
          2 (40, 33, 4000) {1: 20, 2: 20}
          3 (40, 33, 4000) {1: 20, 2: 20}
          4 (40, 33, 4000) {1: 20, 2: 20}
          5 (40, 33, 4000) {1: 20, 2: 20}
          6 (40, 33, 4000) {1: 20, 2: 20}
          7 (40, 33, 4000) {1: 20, 2: 20}
          8 (40, 33, 4000) {1: 20, 2: 20}
          9 (40, 33, 4000) {1: 20, 2: 20}
         10 (40, 33, 4000) {1: 20, 2: 20}
         11 (40, 33, 4000) {1: 20, 2: 20}
         12 (40, 33, 4000) {1: 20, 2: 20}
         13 (40, 33, 4000) {1: 20, 2: 20}
         14 (40, 33, 4000) {1: 20, 2: 20}
         15 (40, 33, 4000) {1: 20, 2: 20}
         16 (40, 33, 4000) {1: 20, 2: 20}
         17 (40, 33, 4000) {1: 20, 2: 20}
         18 (40, 33, 4000) {1: 20, 2: 20}
         19 (40, 33, 4000) {1: 20, 2: 20}
  

## 6. Preprocessing helpers

In [6]:

# Source data layout according to the paper and observed files:
# rawdata: trials x 33 channels x 4000 samples
# Channels 0..29 = EEG-like channels, channel 17 is the CPz reference channel, 30..31 = EOG, 32 = marker.
CHANNEL_SETS = {
    '29eeg_drop_ref_eog_marker': [i for i in range(30) if i != 17],
    '30eeg_include_ref_drop_eog_marker': list(range(30)),
    'motor_region_subset': [12, 13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24],  # Cz/C3/C4/T/CP/P region, 0-based approx
}

WINDOWS = {
    '0_to_4s_paper_literal': (0.0, 4.0),
    '2_to_6s_moabb_interval': (2.0, 6.0),
    '0p5_to_4p5_shifted': (0.5, 4.5),
    '1_to_5_shifted': (1.0, 5.0),
    '1p5_to_5p5_shifted': (1.5, 5.5),
    '0_to_8s_full_trial': (0.0, 8.0),
}

FBCSP_BANDS_COARSE = [(8, 12), (12, 16), (16, 20), (20, 24), (24, 28), (28, 30)]
FBCSP_BANDS_DENSE = [(lo, lo + 4) for lo in range(8, 27)]  # 8-12, 9-13, ..., 26-30


def bandpass_zero_phase(X, sfreq, l_freq, h_freq, order=2):
    sos = butter(order, [l_freq, h_freq], btype='bandpass', fs=sfreq, output='sos')
    return sosfiltfilt(sos, X, axis=-1)


def apply_reference(X, mode):
    if mode == 'none':
        return X
    if mode == 'average':
        return X - X.mean(axis=1, keepdims=True)
    raise ValueError(f'Unknown reference mode: {mode}')


def apply_baseline_removal(X, mode, window_samples=None):
    # X is trials x channels x full trial samples before window crop.
    if mode == 'none':
        return X
    if mode == 'full_trial_mean':
        return X - X.mean(axis=-1, keepdims=True)
    if mode == 'first_2s_mean':
        n = int(round(2.0 * FS))
        return X - X[..., :n].mean(axis=-1, keepdims=True)
    if mode == 'window_mean':
        # Applied later after window crop; leave full trial unchanged here.
        return X
    raise ValueError(f'Unknown baseline mode: {mode}')


def crop_window(X, window_s):
    start_s, stop_s = window_s
    start = int(round(start_s * FS))
    stop = int(round(stop_s * FS))
    if start < 0 or stop > X.shape[-1] or stop <= start:
        raise ValueError(f'Bad window {window_s}; samples=({start}, {stop}), available={X.shape[-1]}')
    return X[..., start:stop]


def prepare_subject_trials(rawdata, labels, variant):
    channel_indices = CHANNEL_SETS[variant['channel_set']]
    X = rawdata[:, channel_indices, :].astype(np.float64)
    y = labels.astype(int).ravel() - 1  # paper labels 1/2 -> 0/1

    # Variant: label flip test only; leave false for paper matching unless needed.
    if variant.get('flip_labels', False):
        y = 1 - y

    # Remove baseline before filtering/cropping, unless window_mean is selected.
    X = apply_baseline_removal(X, variant['baseline_mode'])

    # Optional average reference.
    if variant['reference_mode'] == 'average_before_filter':
        X = apply_reference(X, 'average')

    # Paper says high-pass/filtering and referencing, then 8-30 for quantitative validation.
    # We test direct 8-30 and optional prior 0.5-40 stage.
    if variant.get('pre_band_0p5_40', False):
        X = bandpass_zero_phase(X, FS, 0.5, 40.0, order=variant['filter_order'])

    # Crop window before 8-30 filtering OR after, depending on variant.
    if variant.get('crop_before_bandpass', False):
        X = crop_window(X, WINDOWS[variant['window_name']])
        if variant['baseline_mode'] == 'window_mean':
            X = X - X.mean(axis=-1, keepdims=True)
        X = bandpass_zero_phase(X, FS, variant['band'][0], variant['band'][1], order=variant['filter_order'])
    else:
        X = bandpass_zero_phase(X, FS, variant['band'][0], variant['band'][1], order=variant['filter_order'])
        X = crop_window(X, WINDOWS[variant['window_name']])
        if variant['baseline_mode'] == 'window_mean':
            X = X - X.mean(axis=-1, keepdims=True)

    if variant['reference_mode'] == 'average_after_filter':
        X = apply_reference(X, 'average')

    # Drop any channels that are exactly or nearly flat after preprocessing.
    if variant.get('drop_near_zero_channels', True):
        var = np.var(X, axis=(0, 2))
        keep = var > 1e-20
        X = X[:, keep, :]
        kept_channels = [ch for ch, flag in zip(channel_indices, keep) if flag]
    else:
        kept_channels = channel_indices

    return X, y, kept_channels

## 7. Models

Important change from the earlier paper-mimic notebook: this tests **unregularized CSP** (`reg=None`) because a simple MATLAB CSP baseline is likely closer to the paper than Ledoit-Wolf shrinkage.

In [7]:

def make_csp_lda(n_components=4, reg=None, lda_variant='svd'):
    if lda_variant == 'svd':
        lda = LinearDiscriminantAnalysis(solver='svd')
    elif lda_variant == 'lsqr_auto_shrinkage':
        lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')
    else:
        raise ValueError(f'Unknown lda_variant: {lda_variant}')

    return Pipeline([
        ('csp', CSP(n_components=n_components, reg=reg, log=True, norm_trace=False)),
        ('lda', lda),
    ])


def fbcsp_features_fit_transform(X_train, y_train, X_test, bands, n_components=4, reg=None, filter_order=2):
    train_features = []
    test_features = []
    fitted = []
    for band in bands:
        Xtr_b = bandpass_zero_phase(X_train, FS, band[0], band[1], order=filter_order)
        Xte_b = bandpass_zero_phase(X_test, FS, band[0], band[1], order=filter_order)
        csp = CSP(n_components=n_components, reg=reg, log=True, norm_trace=False)
        train_features.append(csp.fit_transform(Xtr_b, y_train))
        test_features.append(csp.transform(Xte_b))
        fitted.append(csp)
    return np.concatenate(train_features, axis=1), np.concatenate(test_features, axis=1), fitted


def apply_feature_selection(X_train, y_train, X_test, mode='none', k=12, random_state=RANDOM_STATE):
    if mode == 'none':
        return X_train, X_test, None
    k = min(k, X_train.shape[1])
    if mode == 'f_classif':
        selector = SelectKBest(f_classif, k=k)
    elif mode == 'mutual_info':
        selector = SelectKBest(lambda X, y: mutual_info_classif(X, y, random_state=random_state), k=k)
    else:
        raise ValueError(f'Unknown feature selection mode: {mode}')
    Xtr = selector.fit_transform(X_train, y_train)
    Xte = selector.transform(X_test)
    return Xtr, Xte, selector


def run_csp_fold(X_train, y_train, X_test, y_test, model_variant):
    model = make_csp_lda(
        n_components=model_variant['n_components'],
        reg=model_variant['csp_reg'],
        lda_variant=model_variant.get('lda_variant', 'svd'),
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    try:
        scores = model.decision_function(X_test)
    except Exception:
        scores = None
    return y_pred, scores


def run_fbcsp_fold(X_train, y_train, X_test, y_test, model_variant):
    bands = FBCSP_BANDS_DENSE if model_variant['fbcsp_bands'] == 'dense' else FBCSP_BANDS_COARSE
    Xtr_f, Xte_f, _ = fbcsp_features_fit_transform(
        X_train, y_train, X_test,
        bands=bands,
        n_components=model_variant['n_components'],
        reg=model_variant['csp_reg'],
        filter_order=model_variant['filter_order'],
    )
    Xtr_f, Xte_f, selector = apply_feature_selection(
        Xtr_f, y_train, Xte_f,
        mode=model_variant['feature_select'],
        k=model_variant['feature_k'],
    )
    clf = Pipeline([
        ('scaler', StandardScaler()),
        ('svm', SVC(kernel='linear', C=model_variant['svm_C'])),
    ])
    clf.fit(Xtr_f, y_train)
    y_pred = clf.predict(Xte_f)
    try:
        scores = clf.decision_function(Xte_f)
    except Exception:
        scores = None
    return y_pred, scores


def summarize_predictions(y_test, y_pred):
    return {
        'accuracy': accuracy_score(y_test, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
        'kappa': cohen_kappa_score(y_test, y_pred),
        'precision_macro': precision_score(y_test, y_pred, average='macro', zero_division=0),
        'sensitivity_macro': recall_score(y_test, y_pred, average='macro', zero_division=0),
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
    }

## 8. Candidate variants

In [8]:

def base_preproc_variant(**overrides):
    v = {
        'window_name': '0_to_4s_paper_literal',
        'channel_set': '29eeg_drop_ref_eog_marker',
        'baseline_mode': 'full_trial_mean',
        'reference_mode': 'none',  # none, average_before_filter, average_after_filter
        'pre_band_0p5_40': False,
        'crop_before_bandpass': False,
        'filter_order': 2,
        'band': (8.0, 30.0),
        'drop_near_zero_channels': True,
        'flip_labels': False,
    }
    v.update(overrides)
    return v


def base_csp_variant(**overrides):
    v = {
        'model_name': 'CSP_LDA',
        'n_components': 4,
        'csp_reg': None,
        'lda_variant': 'svd',
    }
    v.update(overrides)
    return v


def base_fbcsp_variant(**overrides):
    v = {
        'model_name': 'FBCSP_SVM',
        'n_components': 4,
        'csp_reg': None,
        'filter_order': 2,
        'fbcsp_bands': 'coarse',
        'feature_select': 'none',
        'feature_k': 12,
        'svm_C': 1.0,
    }
    v.update(overrides)
    return v


if RUN_PROFILE == 'focused':
    PREPROCESSING_VARIANTS = [
        # base_preproc_variant(name='paper_0to4_no_ref_unreg'),
        base_preproc_variant(name='paper_0to4_avg_after'),
        base_preproc_variant(name='paper_0to4_avg_before', reference_mode='average_before_filter'),
        base_preproc_variant(name='paper_0to4_no_baseline', baseline_mode='none'),
        base_preproc_variant(name='paper_0to4_window_mean', baseline_mode='window_mean'),
        base_preproc_variant(name='paper_0to4_first2s_mean', baseline_mode='first_2s_mean'),
        base_preproc_variant(name='paper_0to4_30eeg', channel_set='30eeg_include_ref_drop_eog_marker'),
        base_preproc_variant(name='moabb_2to6_no_ref', window_name='2_to_6s_moabb_interval'),
        base_preproc_variant(name='shift_1to5_no_ref', window_name='1_to_5_shifted'),
        base_preproc_variant(name='paper_0to4_preband_0p5_40', pre_band_0p5_40=True),
    ]
    CSP_MODEL_VARIANTS = [
        # base_csp_variant(name='csp_unreg_4comp', csp_reg=None, n_components=4),
        # base_csp_variant(name='csp_unreg_6comp', csp_reg=None, n_components=6),
        # base_csp_variant(name='csp_unreg_8comp', csp_reg=None, n_components=8),
        # base_csp_variant(name='csp_ledoit_4comp', csp_reg='ledoit_wolf', n_components=4),
        # base_csp_variant(name='csp_unreg_4comp_lda_shrink', csp_reg=None, n_components=4, lda_variant='lsqr_auto_shrinkage'),
    ]
    FBCSP_MODEL_VARIANTS = [
        base_fbcsp_variant(name='fbcsp_coarse_unreg_no_fs', fbcsp_bands='coarse', feature_select='none', csp_reg=None),
        base_fbcsp_variant(name='fbcsp_dense_unreg_no_fs', fbcsp_bands='dense', feature_select='none', csp_reg=None),
        base_fbcsp_variant(name='fbcsp_dense_unreg_fclassif_k12', fbcsp_bands='dense', feature_select='f_classif', feature_k=12, csp_reg=None),
        base_fbcsp_variant(name='fbcsp_dense_unreg_fclassif_k24', fbcsp_bands='dense', feature_select='f_classif', feature_k=24, csp_reg=None),
        base_fbcsp_variant(name='fbcsp_dense_ledoit_fclassif_k12', fbcsp_bands='dense', feature_select='f_classif', feature_k=12, csp_reg='ledoit_wolf'),
        base_fbcsp_variant(name='fbcsp_coarse_unreg_C0p1', fbcsp_bands='coarse', feature_select='none', svm_C=0.1, csp_reg=None),
        base_fbcsp_variant(name='fbcsp_coarse_unreg_C10', fbcsp_bands='coarse', feature_select='none', svm_C=10.0, csp_reg=None),
    ]
else:
    PREPROCESSING_VARIANTS = []
    for window_name, channel_set, baseline_mode, reference_mode, pre_band in product(
        ['0_to_4s_paper_literal', '2_to_6s_moabb_interval', '0p5_to_4p5_shifted', '1_to_5_shifted', '1p5_to_5p5_shifted'],
        ['29eeg_drop_ref_eog_marker', '30eeg_include_ref_drop_eog_marker'],
        ['full_trial_mean', 'window_mean', 'none', 'first_2s_mean'],
        ['none', 'average_before_filter', 'average_after_filter'],
        [False, True],
    ):
        PREPROCESSING_VARIANTS.append(base_preproc_variant(
            name=f'{window_name}|{channel_set}|{baseline_mode}|{reference_mode}|preband={pre_band}',
            window_name=window_name,
            channel_set=channel_set,
            baseline_mode=baseline_mode,
            reference_mode=reference_mode,
            pre_band_0p5_40=pre_band,
        ))
    CSP_MODEL_VARIANTS = [
        base_csp_variant(name=f'csp_reg={reg}_comp={n}_lda={lda}', csp_reg=reg, n_components=n, lda_variant=lda)
        for reg, n, lda in product([None, 'ledoit_wolf'], [2, 4, 6, 8], ['svd', 'lsqr_auto_shrinkage'])
    ]
    FBCSP_MODEL_VARIANTS = [
        base_fbcsp_variant(
            name=f'fbcsp_{bands}_reg={reg}_fs={fs_mode}_k={k}_C={C}',
            fbcsp_bands=bands,
            csp_reg=reg,
            feature_select=fs_mode,
            feature_k=k,
            svm_C=C,
        )
        for bands, reg, fs_mode, k, C in product(
            ['coarse', 'dense'], [None, 'ledoit_wolf'], ['none', 'f_classif'], [12, 24], [0.1, 1.0, 10.0]
        )
    ]

log(f'Preprocessing variants: {len(PREPROCESSING_VARIANTS)}')
log(f'CSP model variants:     {len(CSP_MODEL_VARIANTS)}')
log(f'FBCSP model variants:   {len(FBCSP_MODEL_VARIANTS)}')

pd.DataFrame(PREPROCESSING_VARIANTS).to_csv(ARTIFACT_DIR / 'preprocessing_variants.csv', index=False)
pd.DataFrame(CSP_MODEL_VARIANTS).to_csv(ARTIFACT_DIR / 'csp_model_variants.csv', index=False)
pd.DataFrame(FBCSP_MODEL_VARIANTS).to_csv(ARTIFACT_DIR / 'fbcsp_model_variants.csv', index=False)

Preprocessing variants: 9
CSP model variants:     0
FBCSP model variants:   7


## 9. Preload all subjects once

In [9]:

RAW_SUBJECTS = []
for item in subjects:
    rawdata, labels, raw_name, label_name = load_subject_mat(item['path'])
    RAW_SUBJECTS.append({
        'subject_id': item['subject_id'],
        'rawdata': rawdata,
        'labels': labels,
        'path': str(item['path']),
    })

example = RAW_SUBJECTS[0]
log(f"Example raw subject {example['subject_id']}: X={example['rawdata'].shape}, labels={dict(zip(*np.unique(example['labels'], return_counts=True)))}")

Example raw subject 1: X=(40, 33, 4000), labels={np.int64(1): np.int64(20), np.int64(2): np.int64(20)}


## 10. Evaluation loop

In [ ]:

def evaluate_one_combination(preproc_variant, model_variant):
    rows = []
    splitter = StratifiedShuffleSplit(
        n_splits=N_SPLITS,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
    )

    for subj in RAW_SUBJECTS:
        sid = subj['subject_id']
        try:
            X, y, kept_channels = prepare_subject_trials(subj['rawdata'], subj['labels'], preproc_variant)
        except Exception as exc:
            rows.append({
                'subject_id': sid,
                'error': repr(exc),
                'preproc_name': preproc_variant['name'],
                'model_name': model_variant['model_name'],
                'model_variant_name': model_variant['name'],
            })
            continue

        for fold_id, (train_idx, test_idx) in enumerate(splitter.split(X, y), start=1):
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]

            try:
                if model_variant['model_name'] == 'CSP_LDA':
                    y_pred, scores = run_csp_fold(X_train, y_train, X_test, y_test, model_variant)
                elif model_variant['model_name'] == 'FBCSP_SVM':
                    y_pred, scores = run_fbcsp_fold(X_train, y_train, X_test, y_test, model_variant)
                else:
                    raise ValueError(f"Unknown model name: {model_variant['model_name']}")

                metrics = summarize_predictions(y_test, y_pred)
                rows.append({
                    'subject_id': sid,
                    'fold_id': fold_id,
                    'preproc_name': preproc_variant['name'],
                    'model_name': model_variant['model_name'],
                    'model_variant_name': model_variant['name'],
                    'n_train': int(len(train_idx)),
                    'n_test': int(len(test_idx)),
                    'n_channels': int(X.shape[1]),
                    'n_times': int(X.shape[2]),
                    'train_class_counts': np.bincount(y_train, minlength=2).tolist(),
                    'test_class_counts': np.bincount(y_test, minlength=2).tolist(),
                    'prediction_histogram': np.bincount(y_pred, minlength=2).tolist(),
                    **metrics,
                })
            except Exception as exc:
                rows.append({
                    'subject_id': sid,
                    'fold_id': fold_id,
                    'preproc_name': preproc_variant['name'],
                    'model_name': model_variant['model_name'],
                    'model_variant_name': model_variant['name'],
                    'error': repr(exc),
                })

    return rows


all_rows = []
combination_log_rows = []

# CSP sweep
for pi, preproc_variant in enumerate(PREPROCESSING_VARIANTS, start=1):
    for mi, model_variant in enumerate(CSP_MODEL_VARIANTS, start=1):
        log(f"[CSP {pi}/{len(PREPROCESSING_VARIANTS)} | {mi}/{len(CSP_MODEL_VARIANTS)}] {preproc_variant['name']} + {model_variant['name']}")
        rows = evaluate_one_combination(preproc_variant, model_variant)
        all_rows.extend(rows)

        tmp = pd.DataFrame([r for r in rows if 'accuracy' in r])
        if len(tmp):
            mean_acc = tmp['accuracy'].mean()
            log(f"  -> acc={mean_acc:.4f}, distance_to_paper={abs(mean_acc - TARGET_CSP_LDA):.4f}")

# FBCSP sweep
for pi, preproc_variant in enumerate(PREPROCESSING_VARIANTS, start=1):
    for mi, model_variant in enumerate(FBCSP_MODEL_VARIANTS, start=1):
        log(f"[FBCSP {pi}/{len(PREPROCESSING_VARIANTS)} | {mi}/{len(FBCSP_MODEL_VARIANTS)}] {preproc_variant['name']} + {model_variant['name']}")
        rows = evaluate_one_combination(preproc_variant, model_variant)
        all_rows.extend(rows)

        tmp = pd.DataFrame([r for r in rows if 'accuracy' in r])
        if len(tmp):
            mean_acc = tmp['accuracy'].mean()
            log(f"  -> acc={mean_acc:.4f}, distance_to_paper={abs(mean_acc - TARGET_FBCSP_SVM):.4f}")

results_df = pd.DataFrame(all_rows)
results_df.to_csv(ARTIFACT_DIR / 'cv_results.csv', index=False)
log(f'Wrote fold-level results: {ARTIFACT_DIR / "cv_results.csv"}')

[FBCSP 1/9 | 1/7] paper_0to4_avg_after + fbcsp_coarse_unreg_no_fs
Computing rank from data with rank=None
    Using tolerance 6 (2.2e-16 eps * 29 dim * 9.3e+14  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank from 29 -> 29
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 5 (2.2e-16 eps * 29 dim * 7.8e+14  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank from 29 -> 29
Estimating class=0 covariance using EMPIRICAL
Done.
Estimating class=1 covariance using EMPIRICAL
Done.
Computing rank from data with rank=None
    Using tolerance 3.7 (2.2e-16 eps * 29 dim * 5.8e+14  max singular value)
    Estimated rank (data): 29
    data: rank 29 computed from 29 data channels with 0 projectors
Reducing data rank fro

## 11. Rank variants by closeness to paper

In [ ]:

valid_df = results_df[results_df['accuracy'].notna()].copy()

summary = (
    valid_df
    .groupby(['model_name', 'preproc_name', 'model_variant_name'], dropna=False)
    .agg(
        mean_accuracy=('accuracy', 'mean'),
        std_accuracy=('accuracy', 'std'),
        mean_balanced_accuracy=('balanced_accuracy', 'mean'),
        mean_kappa=('kappa', 'mean'),
        mean_precision_macro=('precision_macro', 'mean'),
        mean_sensitivity_macro=('sensitivity_macro', 'mean'),
        n_folds=('accuracy', 'size'),
        mean_n_channels=('n_channels', 'mean'),
        mean_n_times=('n_times', 'mean'),
    )
    .reset_index()
)

summary['target_accuracy'] = np.where(
    summary['model_name'].eq('CSP_LDA'),
    TARGET_CSP_LDA,
    TARGET_FBCSP_SVM,
)
summary['abs_distance_to_paper'] = (summary['mean_accuracy'] - summary['target_accuracy']).abs()
summary = summary.sort_values(['model_name', 'abs_distance_to_paper', 'mean_accuracy'], ascending=[True, True, False])
summary.to_csv(ARTIFACT_DIR / 'paper_match_ranked_summary.csv', index=False)

best_csp = summary[summary['model_name'].eq('CSP_LDA')].head(15)
best_fbcsp = summary[summary['model_name'].eq('FBCSP_SVM')].head(15)

log('')
log('=' * 100)
log('Top CSP+LDA candidates by closeness to paper target 55.57%')
log('=' * 100)
log(best_csp.to_string(index=False))

log('')
log('=' * 100)
log('Top FBCSP+SVM candidates by closeness to paper target 57.57%')
log('=' * 100)
log(best_fbcsp.to_string(index=False))

# Save best-candidate configs for easy inspection.
best_configs = {
    'target_scores': {
        'CSP_LDA': TARGET_CSP_LDA,
        'FBCSP_SVM': TARGET_FBCSP_SVM,
    },
    'best_csp_rows': best_csp.to_dict(orient='records'),
    'best_fbcsp_rows': best_fbcsp.to_dict(orient='records'),
}
with open(ARTIFACT_DIR / 'best_candidate_configs.json', 'w') as f:
    json.dump(best_configs, f, indent=2)

summary.head(30)

## 12. Optional: subject-level view for the best candidate

In [ ]:

def subject_level_for_best(model_name):
    sub = summary[summary['model_name'].eq(model_name)].sort_values('abs_distance_to_paper').head(1)
    if sub.empty:
        return pd.DataFrame()
    row = sub.iloc[0]
    mask = (
        valid_df['model_name'].eq(row['model_name'])
        & valid_df['preproc_name'].eq(row['preproc_name'])
        & valid_df['model_variant_name'].eq(row['model_variant_name'])
    )
    out = (
        valid_df[mask]
        .groupby('subject_id')
        .agg(
            mean_accuracy=('accuracy', 'mean'),
            std_accuracy=('accuracy', 'std'),
            mean_balanced_accuracy=('balanced_accuracy', 'mean'),
            mean_kappa=('kappa', 'mean'),
            n_folds=('accuracy', 'size'),
        )
        .reset_index()
        .sort_values('subject_id')
    )
    out.insert(0, 'model_name', model_name)
    out.insert(1, 'preproc_name', row['preproc_name'])
    out.insert(2, 'model_variant_name', row['model_variant_name'])
    return out

subject_best_csp = subject_level_for_best('CSP_LDA')
subject_best_fbcsp = subject_level_for_best('FBCSP_SVM')
subject_best = pd.concat([subject_best_csp, subject_best_fbcsp], ignore_index=True)
subject_best.to_csv(ARTIFACT_DIR / 'subject_metrics_best_candidates.csv', index=False)
subject_best.head(20)

## 13. Interpretation notes

Use the ranked summary to decide the next fixed notebook:

- If unregularized CSP moves closer to 55.57%, the earlier mismatch was likely CSP regularization rather than data loading.
- If `0_to_4s_paper_literal` wins, the paper likely used the 0–4s source-trial segment literally.
- If `2_to_6s_moabb_interval` wins, MOABB's interval is probably aligned with the MI period even for source trials.
- If average reference wins, update the fixed paper-mimic notebook to reference before/after filtering accordingly.
- If FBCSP only improves with dense bands and feature selection, the paper's FBCSP+SVM was probably not the simple six-band version.

Do not use the highest score blindly if it exceeds the paper by a lot; the goal is to identify the implementation that is most plausible and closest to the reported baseline.